In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
final_prediction_2023_with_test_data = pd.read_excel('./data/final_prediction_2023_with_test_data.xlsx')
rmse_por_llave_resultados_df = pd.read_excel('./data/rmse_por_llave_resultados.xlsx')
data_by_week = pd.read_parquet('./data/almacenes_si_curated_by_week.parquet')
del data_by_week['campaigns_name']

In [ ]:
def get_data_delta_year(df, year:str):
    df_temp = df.copy()
    df_filtred_year = df_temp[df_temp['date_week'].between(f'{year}-01-01',f'{year}-12-31')]
    df_filtred_year['fam_cod'] = df_filtred_year['combination'].apply(lambda x: x[:3])
    df_grouped = df_filtred_year.groupby(['fam_cod'], as_index = False)[['quantity','price_taxes_excluded']].sum()
    df_grouped.sort_values(by = ['price_taxes_excluded'], inplace = True, ascending = False)
    sum_total_price = df_grouped['price_taxes_excluded'].sum()
    df_grouped['price_percentage_total'] = df_grouped['price_taxes_excluded'].apply(lambda x: x/sum_total_price * 100)
    df_grouped['cum_price_percentage_total'] = df_grouped['price_percentage_total'].cumsum()
    return df_grouped
    

In [ ]:
df_2023 = get_data_delta_year(data_by_week, year = '2023')
df_2023.head(1)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def pareto_plot(df):
    df = df.copy()
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Scatter(x=df['fam_cod'], y=df['cum_price_percentage_total'], mode="lines"),
        secondary_y=True
    )

    fig.add_trace(
        go.Bar(x=df['fam_cod'], y=df['price_taxes_excluded'], marker_color=px.colors.qualitative.Dark24),
        secondary_y=False
    )

    fig.update_xaxes(title_text="Letter")

    # Set y-axes titles
    fig.update_yaxes(title_text="Total Vendido", secondary_y=False)
    fig.update_yaxes(title_text="Porcentage Acumulado", secondary_y=True)
    fig.update_layout(showlegend=False)
    fig.update_layout(title='Pareto Chart', xaxis_title='Codigo Familia')
    fig.show()
    return fig

In [ ]:
pareto_plot(df_2023)

In [ ]:
df_2023

In [ ]:
from utils import pareto_plot

In [ ]:
pareto_plot(df_2023)

In [ ]:
family_codes_pareto = df_2023[df_2023['cum_price_percentage_total'] <= 80]['fam_cod'].values
family_codes_pareto

In [ ]:
df_filtred_year = data_by_week[data_by_week['date_week'].between(f'2023-01-01',f'2023-12-31')]
df_filtred_year['fam_cod'] = df_filtred_year['combination'].apply(lambda x: x[:3])

In [ ]:
rmse_por_llave_resultados_df['fam_cod'] = rmse_por_llave_resultados_df['llave'].apply(lambda x: x[:3])
rmse_por_llave_resultados_df[rmse_por_llave_resultados_df['fam_cod'].isin(family_codes_pareto)]

In [ ]:
proporcion_error_porcentual = pd.read_excel('./data/proporcion_error_porcentual.xlsx')

In [ ]:
proporcion_error_porcentual['proporcion_error_porcentual'].describe()

In [ ]:
quantile_95 = np.quantile(proporcion_error_porcentual['proporcion_error_porcentual'], .90)
quantile_95

In [ ]:
proporcion_error_porcentual[proporcion_error_porcentual['proporcion_error_porcentual'] > quantile_95]['llave'].values

In [ ]:
def get_data_for_plot_historical_info(data_by_week, predictions_2023_dataset, llave):
    x = data_by_week[data_by_week['combination'] == llave]
    x['month_year'] = x['date_week'].apply(lambda x: x.strftime('%m-%Y'))
    x = x.groupby(['month_year'], as_index=False)['quantity'].sum()
    x['month_year'] = pd.to_datetime(x['month_year'])
    x.sort_values(by = ['month_year'], inplace=True)
    y_true_historical = x.copy()
    
    
    
    y = predictions_2023_dataset[predictions_2023_dataset['llave'] == llave]
    y['month_year'] = y['ds'].apply(lambda x: x.strftime('%m-%Y'))
    y = y.groupby(['month_year'], as_index=False)['yhat'].sum()
    y['month_year'] = pd.to_datetime(y['month_year'])
    y.sort_values(by = ['month_year'], inplace=True)
    y_hat_2023 = y.copy()
    
    return y_true_historical,y_hat_2023

def plot_line_plot_ytrue_hat(y_true_historical,y_hat_2023, llave):
    import plotly.graph_objects as go


    # Create traces
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y_true_historical['month_year'], y=y_true_historical['quantity'],
                        mode='lines+markers',
                        name='lines+markers',line = dict(color='#414141', width=2,)))
    fig.add_trace(go.Scatter(x=y_hat_2023['month_year'], y=y_hat_2023['yhat'],
                        mode='lines+markers',
                        name='lines+markers',line = dict(color='#FF407D', width=2,)))
    fig.update_layout(showlegend=False)

    fig.update_layout(title=f'Comportamiento historico<br>Llave: {llave}',
                    xaxis_title='Month-Year',
                    yaxis_title='Total Venta')
    fig.add_vline(x=pd.to_datetime('01-2023'), line_width=1.5, line_dash="dash", line_color="#FF407D")

    return fig
    
    

In [ ]:
llave = '209489BG9'
x = data_by_week[data_by_week['combination'] == llave]
# x = x[x['date_week'] < '2023-01-01']
x['month_year'] = x['date_week'].apply(lambda x: x.strftime('%m-%Y'))
x = x.groupby(['month_year'], as_index=False)['quantity'].sum()
x['month_year'] = pd.to_datetime(x['month_year'])
x.sort_values(by = ['month_year'], inplace=True)

In [ ]:

llave = '209489BG9'
y = final_prediction_2023_with_test_data[final_prediction_2023_with_test_data['llave'] == llave]
y['month_year'] = y['ds'].apply(lambda x: x.strftime('%m-%Y'))
y = y.groupby(['month_year'], as_index=False)['yhat'].sum()
y['month_year'] = pd.to_datetime(y['month_year'])
y.sort_values(by = ['month_year'], inplace=True)
y

In [ ]:
import plotly.graph_objects as go


# Create traces
fig = go.Figure()
fig.add_trace(go.Scatter(x=x['month_year'], y=x['quantity'],
                    mode='lines+markers',
                    name='lines+markers',line = dict(color='#414141', width=2,)))
fig.add_trace(go.Scatter(x=y['month_year'], y=y['yhat'],
                    mode='lines+markers',
                    name='lines+markers',line = dict(color='#FF407D', width=2,)))
fig.update_layout(showlegend=False)

fig.update_layout(title=f'Comportamiento historico<br>Llave: {llave}',
                   xaxis_title='Month-Year',
                   yaxis_title='Total Venta')
fig.add_vline(x=pd.to_datetime('01-2023'), line_width=1.5, line_dash="dash", line_color="#FF407D")

fig.show()